# Conducto 1D estacionario (Calbuco) — **sin IDA**

Misma física que `RIconduitex5_5.py`, pero la marcha en profundidad $z$ usa **RK4 con paso adaptativo** (método de líneas / diferencias finitas en $z$). Solo hace falta **numpy** y **matplotlib** (`requirements-simple.txt`).

Módulo: `RIconduitex5_5_march.py` → función `RIconduitex5_5_march_f`.

### Dependencias

```bash
pip install -r requirements-simple.txt
```

Ejecutar todas las celdas desde la carpeta del repositorio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from calbuco2015d import radius1, overP1, h2o1, T1, xi1, H, pfinal
from RIconduitex5_5_march import RIconduitex5_5_march_f, perfil_para_grafico

## Caso Calbuco: método de tiro + marcha RK4

Puede tardar ~10–30 s según la máquina (varias iteraciones de tiro).

In [ ]:
radius = radius1
Pressure = overP1

(
    zsol,
    sol,
    count,
    vinicial,
    rho_m,
    rbub,
    visc,
    rho_ti,
    fragcrit,
    phicrit,
    limphi1,
    limphi2,
    co,
    xi,
    xfinal,
    cg,
) = RIconduitex5_5_march_f(radius, Pressure, h2o1, T1, xi1)

In [ ]:
z = np.asarray(zsol, dtype=float)
Y = np.asarray(sol, dtype=float)

print(f"Iteraciones de tiro: {count}")
print(f"Velocidad en la cámara (convergida): {vinicial:.3f} m/s")
print(f"Profundidad boca: {z[-1]:.1f} m  (H cámara = {H:.0f} m)")
print(f"Presión boca: {Y[-1, 0]/1e6:.4f} MPa  (pfinal ≈ {pfinal/1e6:.4f} MPa)")
print(f"φ boca: {Y[-1, 1]:.4f}  |  u_g boca: {Y[-1, 5]:.2f} m/s")

## Perfiles a lo largo del conducto

El solver **sin IDA** puede mostrar un salto artificial en φ (p. ej. 0 → 0.99) al pasar del tramo **líquido** (φ=0) al tramo **burbujeante** (~−3.6 km): el RK4 explícito es rígido ahí. Para las figuras usamos `perfil_para_grafico`, que corrige φ con el valor de **equilibrio** ~P (la presión y velocidades siguen siendo las del modelo).

La versión con **IDA** no suele mostrar ese salto; para comparación estricta con el paper use `RIconduitex5_5.py`.

In [ ]:
z_plot, Y_plot = perfil_para_grafico(zsol, sol, co_l=co, xi_l=xi, rho_m_l=rho_m)

fig, axs = plt.subplots(1, 6, figsize=(16, 7), sharey=True)

axs[0].semilogx(Y_plot[:, 0], z_plot, linewidth=2, color="C0")
axs[0].set_xlabel("Presión (Pa)")
axs[0].set_ylabel("Profundidad z (m)")
axs[0].grid(True, alpha=0.3)

axs[1].plot(Y_plot[:, 1], z_plot, linewidth=2, color="C1", label="φ (suavizado)")
axs[1].plot(Y[:, 1], z, linewidth=1, color="C1", alpha=0.35, ls=":", label="φ (crudo)")
axs[1].set_xlabel("Fracción gas φ")
axs[1].axvline(limphi1, color="gray", ls=":", label="limφ₁")
axs[1].axvline(phicrit, color="k", ls="--", label="φ_crit")
axs[1].legend(fontsize=8)
axs[1].grid(True, alpha=0.3)

axs[2].semilogx(Y_plot[:, 4], z_plot, label="u_m", linewidth=2)
axs[2].semilogx(Y_plot[:, 5], z_plot, "--", label="u_g", linewidth=2)
axs[2].set_xlabel("Velocidad (m/s)")
axs[2].legend(fontsize=8)
axs[2].grid(True, alpha=0.3)

axs[3].plot(Y_plot[:, 2], z_plot, linewidth=2, color="C3")
axs[3].set_xlabel("N_d (1/m³)")
axs[3].grid(True, alpha=0.3)

axs[4].plot(Y_plot[:, 3], z_plot, linewidth=2, color="C4")
axs[4].set_xlabel("Contenido cristal x")
axs[4].grid(True, alpha=0.3)

visc_plot = np.interp(z_plot, z, visc)
axs[5].semilogx(visc_plot, z_plot, linewidth=2, color="C5")
axs[5].set_xlabel("Viscosidad (Pa·s)")
axs[5].grid(True, alpha=0.3)

fig.suptitle("Calbuco — RIconduitex5_5_march (RK4, sin IDA)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## (Opcional) Un tramo con `solv`: marcha explícita en $z$

Útil para una tarea: fijar condiciones iniciales y avanzar un tramo sin método de tiro.

In [ ]:
import RIconduitex5_5_march as M
from calbuco2015d import R, beta, C1, g, rcrust, limperl, limperh
from density import density
from calbuco2015d import sio2, tio2, al2o3, feo, mgo, cao, na2o, k2o

M.h2o = h2o1
M.wr = float(radius1)
M.T = float(T1)
M.Tc = T1 - 273.15
M.co = h2o1 / 100.0
M.xi = float(xi1)
M.cg = 8
Pi = rcrust * g * abs(H) + overP1
M.Pi = Pi
dis = min(C1 * Pi ** beta, M.co)
M.rho_m = density(sio2, tio2, al2o3, feo, mgo, cao, na2o, k2o, dis * 100, M.Tc, Pi / 1e6)
M.q = 10.0 * M.rho_m
M.vinicial = 10.0
M.Nd = 1e8
M.xfinal = xi1
M.n_eq = 1
M.phicrit = 0.75
M.limphi1 = limperl
M.limphi2 = limperh
M.xmax = 0.5

y0 = np.array([Pi, 0.05, M.Nd, M.xi, 10.0, 10.0])
Y_seg, Z_seg = M.solv(H, -2500.0, y0, None, 1e-6, 1e-6, 400, 1, cond=None)

fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
ax[0].semilogx(Y_seg[:, 0], Z_seg, "b-")
ax[0].set_xlabel("P (Pa)")
ax[0].set_ylabel("z (m)")
ax[0].set_title("Tramo corto desde la cámara")
ax[0].grid(True, alpha=0.3)
ax[1].plot(Y_seg[:, 1], Z_seg, "r-")
ax[1].set_xlabel("φ")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Puntos en la malla: {len(Z_seg)}  |  z: {Z_seg[0]:.0f} → {Z_seg[-1]:.0f} m")